In [4]:
#imports

from tensorzinb.tensorzinb import TensorZINB
#import scMPRAforge as scm

TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [5]:
import pandas as pd
import numpy as np
import time
import pickle
from formulaic import Formula
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
#create dask cluster

from dask_jobqueue import SLURMCluster
from dask.distributed import Client

cluster=SLURMCluster(
    cores=4,#cores per slurm job
    memory="32G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p day", 
        f"--job-name=simclust_worker",
        f"--time=3:00:00",
        f"--output=worker_%j.out"]
)

cluster.scale(jobs=3)

client = Client(cluster,
        timeout=f"{5*60}s",   # Client <-> scheduler timeout 
        heartbeat_interval="20s"  # Worker heartbeat interval
    )

#from dask.distributed import Client, LocalCluster
#cluster=LocalCluster(memory_limit='8GB')
#client = Client(cluster)

In [4]:
client.dashboard_link

NameError: name 'client' is not defined

# Describe with Ortho


In [ ]:
data_root="/home/sxl6/project_pi_mg269/sxl6/tabula_data/seelig"
path= data_root
name="ortho_test_seelig"

import os
if os.path.isdir(path+"/"+name):
    print("[+] Model found. Loading...")
    primordial=scm.ortho.load(client,path,name)
    seelig=primordial.training_data
else:
    print("[+] Model not found. Creating...")

    #load data
    seelig=scm.scMPRA_data.from_tsv(f"{data_root}/seelig_counts_grouped.txt")
    # place holder for now , may need to change if ask seelig paper author what they used as negative control/ how they selected them exactly
    seelig.set_negative_controls(["AACGCCCTCCACGGATGGGCCGGCCAATAAGAAGCGTTAGCGGACTCATGCGTTACGCGCCTCCGAGTTATGGGGGGGGAGGCGCGTATCTCGTGGAGAAGAAGCGATGTAACGCTTGGGCGATAAGCTTATAAGGAAGATATTT",
    "CCCTCGGAGTTAATAAGATACGCGGATCGATATCGGCTTGAAGAAGCGTATCTTATCTTCAGATGGGGATGTCGCGCATCCACCCAGTGGGCACCGCCGCTATAGAAGGGTGATAACGCTTCTCAGCCTTCAGGCTCTGGGTCTT"])
    seelig.set_reference_cell("HEPG2")
    seelig.ortho_filter()

    primordial=scm.ortho()
    primordial.criss_cross(client=client,
                       dat=seelig)
    primordial.extract_params(client)
    primordial.save(path,name)


[+] Model not found. Creating...


scMPRAforge: INFO: Dropped 81 of 2688 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
